In [ ]:
def flir_video_python(cam, initial_exposure_ms, roi_params, max_exposure_ms):
    """
    Args:
        cam: PySpin Camera object
        initial_exposure_ms: Initial exposure time in ms
        roi_params: [x_offset, y_offset, width, height]
        max_exposure_ms: Maximum limit for the slider
    """
    # 1. Setup Camera ROI and Initial Exposure
    nodemap = cam.GetNodeMap()
    
    # Set ROI (Region of Interest)
    cam.Width.SetValue(roi_params[2])
    cam.Height.SetValue(roi_params[3])
    cam.OffsetX.SetValue(roi_params[0])
    cam.OffsetY.SetValue(roi_params[1])
    
    # Set Exposure Mode to Manual
    cam.ExposureAuto.SetValue(PySpin.ExposureAuto_Off)
    cam.ExposureTime.SetValue(initial_exposure_ms * 1000.0)  # Spinnaker uses microseconds

    # 2. Setup Matplotlib GUI
    fig, ax = plt.subplots()
    plt.subplots_adjust(bottom=0.25)
    
    # Initial Acquisition to get frame size
    cam.BeginAcquisition()
    image_result = cam.GetNextImage()
    img_data = image_result.GetNDArray()
    im_plot = ax.imshow(img_data, cmap='viridis')
    image_result.Release()
    
    # Add Slider Axis [left, bottom, width, height]
    ax_slider = plt.axes([0.1, 0.1, 0.65, 0.03])
    val_min = 0.01  # 11ms in your code, but usually lower for FLIR
    slider = Slider(ax_slider, 'Exp (ms)', val_min, max_exposure_ms, valinit=initial_exposure_ms)

    # 3. Define Update Logic
    current_exposure = initial_exposure_ms

    def update_exposure(val):
        nonlocal current_exposure
        current_exposure = slider.val
        # Update camera hardware in real-time
        try:
            cam.ExposureTime.SetValue(current_exposure * 1000.0)
        except PySpin.SpinnakerException as e:
            print(f"Error setting exposure: {e}")

    slider.on_changed(update_exposure)

    # 4. Main Acquisition Loop
    print("Press [Esc] on the figure or close the window to stop.")
    try:
        while plt.fignum_exists(fig.number):
            image_result = cam.GetNextImage(1000)  # Timeout 1s
            
            if image_result.IsIncomplete():
                print(f"Image incomplete: {image_result.GetImageStatus()}")
            else:
                data = image_result.GetNDArray()
                im_plot.set_data(data)
                
                # Optional: Auto-scale intensity if needed
                # im_plot.set_clim(vmin=np.min(data), vmax=np.max(data))
                
                fig.canvas.draw_idle()
                plt.pause(0.001)  # Allow GUI events to process
            
            image_result.Release()
            
    except KeyboardInterrupt:
        pass
    finally:
        # Cleanup
        cam.EndAcquisition()
        return current_exposure / 1000.0  # Return in seconds as per original code